# EDA-0 — Study Construction Integrity

Confirm the study dataset was built correctly on top of the DAL.
This notebook assumes the 25 DAL integrity tests pass — it does not re-test DAL invariants.
It checks only what the DAL cannot guarantee: rolling window construction, lag alignment,
and activity filter population.

## Scope

- Study range: GW 6 to GW 34
- Inputs: curated spine and state outputs
- Gate: if Q0.1 or Q0.2 fails, stop before EDA-1
- Q0.3 gate check: if any position-GW block is empty after filter, stop
- Note: Detailed filter analysis (removal rates, Y effects) moves to EDA-3

## Notebook Setup

Imports, constants, and helpers needed before any check can run.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from dal.curated.player_gameweek_spine import build_player_gameweek_spine
from dal.state.player_gameweek_state import build_player_gameweek_state
from research.eda.notebooks._integrity_helpers import (
    build_findings_template,
    check_activity_filter_gate,
    check_lag_alignment,
    check_rolling_windows,
)
from research.eda.notebooks._target_distribution_helpers import analyze_data_completeness

DB_PATH = Path.home() / ".fpl" / "fpl.db"
STUDY_GW_MIN = 6
STUDY_GW_MAX = 34
CHECK_GWS = [8, 15, 25]
POSITION_PICK = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
ACTIVITY_MIN = 45

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## Load Inputs

Build the full spine and state tables, then filter to the study GW range.
State_all is kept unfiltered so rolling windows can be verified against full history.

In [ ]:
spine_all = build_player_gameweek_spine(DB_PATH)
state_all = build_player_gameweek_state()

state = state_all[state_all["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()

print(f"DB_PATH: {DB_PATH}")
print(f"State rows (GW {STUDY_GW_MIN}-{STUDY_GW_MAX}): {len(state):,}")
display(state.head())

## Result Recording Helpers

In [ ]:
results: dict[str, dict[str, object]] = {}


def record_result(question: str, result: str, **details: object) -> None:
    results[question] = {"result": result, **details}


def print_header(title: str) -> None:
    print(f"\n{'=' * 88}")
    print(title)
    print(f"{'=' * 88}")

## Q0.1 — Rolling Window Construction

**Question:** Are roll-3 and roll-5 values correct for the study population?

Spot-check 3 players (one GK, one MID, one FWD) with continuous history across GW 1–25
at checkpoints GW 8, 15, and 25. Also check an edge-case player with gaps in fixture history.

In [ ]:
print_header("Q0.1 — Rolling window construction")

q01 = check_rolling_windows(
    state_all=state_all,
    position_pick=POSITION_PICK,
    check_gws=CHECK_GWS,
)
verification_df = q01["verification_df"]
display(verification_df)
display(q01["verification_table"])
if q01["edge_case_table"].empty:
    print(
        "\n⚠ WARNING: No player with fixture gaps found in the validation window.\n"
        "  The edge-case rolling window check (gaps in history) was not executed.\n"
        "  Manually identify a player with known missing GWs and verify roll values\n"
        "  before treating Q0.1 as fully PASS."
    )
else:
    print("Edge-case player with fixture gaps:")
    display(q01["edge_case_table"].head(12))
record_result(
    "Q0.1",
    q01["result"],
    players_verified=q01["players_verified"],
    gws_checked=q01["gws_checked"],
    mismatches=q01["mismatches"],
    edge_case_verified=not q01["edge_case_table"].empty,
)

## Q0.2 — Lag Alignment

**Question:** For lag-1 signals, is X at GW N correctly paired with Y at GW N+1?

Spot-check the same 3 verification players at the same GW checkpoints.
Confirm the shift-based lag construction matches a direct GW N+1 lookup.

In [ ]:
print_header("Q0.2 — Lag alignment")

q02 = check_lag_alignment(
    state_all=state_all,
    verification_df=verification_df,
    check_gws=CHECK_GWS,
)
print("Columns containing 'next':", q02["next_cols"])
display(q02["lag_table"])
record_result(
    "Q0.2",
    q02["result"],
    approach=q02["approach"],
    players_verified=q02["players_verified"],
    mismatches=q02["mismatches"],
)

## Q0.3 — Activity Filter Gate Check

**Question:** After applying the activity filter, does every position-GW block cell have records?

**Gate function:** Confirm the filter does not inadvertently empty any stratum.
Detailed analysis of filter effects (removal rates, Y distribution) belongs in EDA-3.

In [ ]:
print_header("Q0.3 — Activity filter gate check")

q03 = check_activity_filter_gate(state, activity_min=ACTIVITY_MIN)
print(f"Activity filter: minutes >= {q03['activity_min']}")
print(f"\nRecords per position per GW block:")
display(q03["counts_table"])
print(f"\nEmpty cells: {q03['empty_cells']}")
record_result(
    "Q0.3",
    q03["result"],
    activity_min=q03["activity_min"],
    empty_cells=q03["empty_cells"],
)

## Q0.4 — Study Population Completeness

**Question:** How many rows are excluded from the study population, and why?

**Why this matters:** The `all` cohort in EDA-1 is filtered to `starts.notna()`. This check confirms that excluded rows are exclusively BGW scaffold rows — structural absences where no fixture existed — not real playing records. It also surfaces late joiners whose first appearance is after GW 6, which affects rolling window availability.

**Gate:** If any excluded row has `minutes > 0` or `total_points > 0`, the filter is wrong and must be corrected before EDA-1.

In [ ]:
print_header("Q0.4 — Study Population Completeness")

completeness = analyze_data_completeness(state_all, study_gw_min=STUDY_GW_MIN)

print("Data Completeness Report")
print("=" * 60)
print(f"Total rows in full state:        {completeness['total_rows']:,}")
print(f"Null 'started' rows excluded:    {completeness['null_rows']:,} ({completeness['null_pct']:.1f}%)")
print(f"Late joiners (first_gw > {STUDY_GW_MIN}):    {completeness['late_joiners']}")

print(f"\nFirst appearance distribution (players who joined study):")
first_gws = completeness['first_appearances']
print(first_gws.value_counts().sort_index().head(10))

print(f"\n→ Analysis uses only non-null rows (study range GW {STUDY_GW_MIN}-{STUDY_GW_MAX})")

## Summary For EDA_FINDINGS.md

Overall gate decision and ready-to-copy findings template.

In [ ]:
summary = pd.DataFrame(
    [{"question": question, "result": payload["result"]} for question, payload in results.items()]
)
display(summary)

hard_fail = any(results[q]["result"] == "FAIL" for q in ["Q0.1", "Q0.2", "Q0.3"])
gate_decision = "STOP" if hard_fail else "PROCEED TO EDA-1"
overall_result = "FAIL" if hard_fail else "PASS"

print_header("EDA-0 summary")
print(f"Overall result: {overall_result}")
for question in ["Q0.1", "Q0.2", "Q0.3"]:
    print(f"{question}: {results[question]['result']}")
print(f"Gate decision: {gate_decision}")

print("\nTemplate for EDA_FINDINGS.md:\n")
for line in build_findings_template(results, overall_result, gate_decision):
    print(line)